# 1. Workspace and model artifacts

This tutorial introduces the filesystem workspace, local image contexts, and portable model storage. The examples assume that the package is installed with `pip install -e .` from the repository root.

## Download the example bundle

The final bundle will contain an imzML/ibd image pair, a latent imzML/ibd pair, and a saved model directory. Replace the placeholder URL when the release artifact is published, then uncomment the shell commands.

In [ ]:
TUTORIAL_BUNDLE_URL = "TODO: add the published tutorial bundle URL"
# !wget -O msi_tutorial_bundle.zip "$TUTORIAL_BUNDLE_URL"
# !unzip -q msi_tutorial_bundle.zip -d msi_tutorial_bundle

## Create a workspace

A workspace is the persistent project root. By default it contains `imgs/` and `models/`. Creating an empty workspace is valid, but a reader cannot be initialized until its `.imzML` file exists. The matching `.ibd` file must remain beside it.

In [ ]:
from pathlib import Path
from msi_autoencoder_wrapper.core.wrapper import MSIAutoEncoderWrapper

workspace_root = Path("tutorial_workspace").resolve()
wrapper = MSIAutoEncoderWrapper(
    project_path=str(workspace_root),
    device="cpu",
    coordinate_order="xy",
)
print(wrapper.workspace.project_path_resolved)
print(wrapper.workspace.get_imgs_dir())

Use a custom layout only when another directory convention is required. Keep all four keys because model loading reconstructs paths from this mapping.

In [ ]:
custom_layout = {
    "imgs_dir": "data/images",
    "models_root": "artifacts/models",
    "model_config_subdir": "configuration",
    "model_latent_subdir": "latent",
}
custom_wrapper = MSIAutoEncoderWrapper(
    project_path="custom_workspace",
    layout=custom_layout,
)

## Add and select an image

Copy both files before configuring the reader. Passing a bare image key resolves to `<workspace>/imgs/<key>.imzML`; an absolute imzML path may point outside the workspace. Selecting an image changes the active local context, but does not load data by itself.

In [ ]:
import shutil

bundle_image = Path("msi_tutorial_bundle/example.imzML")
bundle_binary = bundle_image.with_suffix(".ibd")
images_dir = wrapper.workspace.get_imgs_dir()
# Run after downloading the bundle:
# shutil.copy2(bundle_image, images_dir / bundle_image.name)
# shutil.copy2(bundle_binary, images_dir / bundle_binary.name)
wrapper.workspace.set_active_image("example")
print(wrapper.workspace.get_active_image_file_path())

## Load, bind, save, and export a model

The model manager owns exactly one currently loaded model. `bind_to_local_context=True` additionally keeps its functionality in the active image context. A later call to `load_model` replaces the manager's loaded model but does not overwrite that local binding. Saved models use JSON configuration plus a PyTorch state dictionary; full pickled modules are not the default portable format.

In [ ]:
# The bundle must be extracted into the matching workspace model layout first.
# loaded = wrapper.models_manager.load_model(
#     img_name="example",
#     model_name="example-autoencoder",
#     bind_to_local_context=True,
# )
# print(wrapper.models_manager.loaded_model is loaded)
# print(wrapper.models_manager.model_functionality)
# print(wrapper.active_context.local_model_functionality)

In [ ]:
# Save the currently loaded model under the active image context.
# model_dir = wrapper.workspace.save_model(model_name="example-autoencoder")
# Export is explicit and copies the complete saved folder.
# exported_dir = wrapper.workspace.export_model_folder(
#     destination="exports/example-autoencoder",
#     model_name="example-autoencoder",
# )
# print(model_dir, exported_dir)

Use `img_name="global"` only for a model that is intentionally independent of one image context. Global and multi-image training contracts are still planned; current tutorials focus on local single-image models. Continue with [Tutorial 2](02_readers_binners_and_coordinates.ipynb).